In [150]:
import pandapipes as pp
from pandapipes.pf.pipeflow_setup import get_fluid
import pandas as pd
import numpy as np

In [ ]:
import pandapipes as pp
from pandapipes.pf.pipeflow_setup import get_fluid
import pandas as pd
import numpy as np

class ThermoclineTwoLayer():
    def __init__(self, net, name, circ_pump_charge_id, circ_pump_decharge_id, 
                 flow_control_charge_id, flow_control_decharge_id, 
                 flow_control_bypass_charge_go_id, flow_control_bypass_charge_return_id,
                 flow_control_bypass_decharge_go_id, flow_control_bypass_decharge_return_id,
                 volume_m3, t_hot_init, t_cold_init, v_hot_fraction_init, UA_loss, UA_interface=0.0):
        
        self.net = net
        self.fluid = get_fluid(net)
        self.name = name
        self.circ_pump_charge_id = circ_pump_charge_id
        self.circ_pump_decharge_id = circ_pump_decharge_id
        self.flow_control_charge_id = flow_control_charge_id
        self.flow_control_decharge_id = flow_control_decharge_id
        self.flow_control_bypass_charge_go_id = flow_control_bypass_charge_go_id
        self.flow_control_bypass_charge_return_id = flow_control_bypass_charge_return_id
        self.flow_control_bypass_decharge_go_id = flow_control_bypass_decharge_go_id
        self.flow_control_bypass_decharge_return_id = flow_control_bypass_decharge_return_id
        
        self.V_tot = volume_m3
        self.T_hot = t_hot_init
        self.T_cold = t_cold_init
        self.v_hot_fraction = v_hot_fraction_init
        self.UA_loss = UA_loss
        self.UA_interface = UA_interface
        self.V_hot = volume_m3 * self.v_hot_fraction
        self.V_cold = volume_m3 - self.V_hot
        self.V_MIN = 1e-4

        # Inicialización de variables operativas
        self.bypass = False
        self.direction = None
        self.mass_flow = 0.0
        self.mdot_entering = 0.0
        self.mdot_bypass = 0.0

    def evaluate_bypass(self, T_network, mass_flow, dt_s): 
        self.mass_flow = mass_flow
        bypass = False
        direction = None
        mdot_entering = 0.0
        mdot_bypass = 0.0

        if mass_flow >= 0: # Carga
            if T_network < self.T_hot:
                bypass = True
                direction = 'to_Tcold'
                mdot_entering = 0.0
                mdot_bypass = mass_flow
            else:
                density = self.fluid.get_density(self.T_hot)
                Dv_requested = (mass_flow * dt_s) / density
                v_available = max(0.0, self.V_cold - self.V_MIN)
                v_in = min(Dv_requested, v_available)

                mdot_entering = (v_in * density) / dt_s if dt_s > 0 else 0.0
                mdot_bypass = mass_flow - mdot_entering
                if mdot_bypass > 1e-6 or self.V_cold <= self.V_MIN:
                    bypass = True
        else: # Descarga
            abs_mass_flow = abs(mass_flow)
            density = self.fluid.get_density(self.T_cold)
            Dv_requested = (abs_mass_flow * dt_s) / density
            v_available = max(0.0, self.V_hot - self.V_MIN)
            v_in = min(Dv_requested, v_available)

            mdot_entering = (v_in * density) / dt_s if dt_s > 0 else 0.0
            mdot_bypass = abs_mass_flow - mdot_entering
            if mdot_bypass > 1e-6 or self.V_hot <= self.V_MIN:
                bypass = True

        self.bypass = bypass
        self.direction = direction
        self.mdot_entering = mdot_entering
        self.mdot_bypass = mdot_bypass

    def apply_control_settings(self):
        """Aplica los caudales calculados a los componentes de pandapipes"""
        if self.bypass:
            if self.mass_flow >= 0:    
                    self.net.flow_control.at[self.flow_control_bypass_charge_go_id, "controlled_mdot_kg_per_s"] = self.mdot_bypass
                    self.net.flow_control.at[self.flow_control_bypass_charge_return_id, "controlled_mdot_kg_per_s"] = self.mdot_bypass
                    self.net.flow_control.at[self.flow_control_bypass_decharge_go_id, "controlled_mdot_kg_per_s"] = 0
                    self.net.flow_control.at[self.flow_control_bypass_decharge_return_id, "controlled_mdot_kg_per_s"] = 0
                    
                    self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "mdot_flow_kg_per_s"] = 0
                    self.net.flow_control.at[self.flow_control_decharge_id, "controlled_mdot_kg_per_s"] = 0
                    self.net.circ_pump_mass.at[self.circ_pump_charge_id, "mdot_flow_kg_per_s"] = self.mdot_entering
                    self.net.circ_pump_mass.at[self.circ_pump_charge_id, "t_flow_k"] = self.T_cold
                    self.net.flow_control.at[self.flow_control_charge_id, "controlled_mdot_kg_per_s"] = self.mdot_entering
            else:
                self.net.flow_control.at[self.flow_control_bypass_charge_go_id, "controlled_mdot_kg_per_s"] = self.mdot_bypass
                self.net.flow_control.at[self.flow_control_bypass_charge_return_id, "controlled_mdot_kg_per_s"] = self.mdot_bypass
                self.net.flow_control.at[self.flow_control_bypass_decharge_go_id, "controlled_mdot_kg_per_s"] = 0
                self.net.flow_control.at[self.flow_control_bypass_decharge_return_id, "controlled_mdot_kg_per_s"] = 0
                
                self.net.circ_pump_mass.at[self.circ_pump_charge_id, "mdot_flow_kg_per_s"] = 0
                self.net.flow_control.at[self.flow_control_charge_id, "controlled_mdot_kg_per_s"] = 0
                self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "mdot_flow_kg_per_s"] = self.mdot_entering  # Corregido typo
                self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "t_flow_k"] = self.T_hot
                self.net.flow_control.at[self.flow_control_decharge_id, "controlled_mdot_kg_per_s"] = self.mdot_entering
        else:
            # Desactivar bypass
            self.net.flow_control.at[self.flow_control_bypass_charge_go_id, "controlled_mdot_kg_per_s"] = 0
            self.net.flow_control.at[self.flow_control_bypass_charge_return_id, "controlled_mdot_kg_per_s"] = 0
            self.net.flow_control.at[self.flow_control_bypass_decharge_go_id, "controlled_mdot_kg_per_s"] = 0
            self.net.flow_control.at[self.flow_control_bypass_decharge_return_id, "controlled_mdot_kg_per_s"] = 0

            if self.mass_flow >= 0:
                self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "mdot_flow_kg_per_s"] = 0
                self.net.flow_control.at[self.flow_control_decharge_id, "controlled_mdot_kg_per_s"] = 0
                self.net.circ_pump_mass.at[self.circ_pump_charge_id, "mdot_flow_kg_per_s"] = self.mass_flow
                self.net.circ_pump_mass.at[self.circ_pump_charge_id, "t_flow_k"] = self.T_cold
                self.net.flow_control.at[self.flow_control_charge_id, "controlled_mdot_kg_per_s"] = self.mass_flow
            else:
                self.net.circ_pump_mass.at[self.circ_pump_charge_id, "mdot_flow_kg_per_s"] = 0
                self.net.flow_control.at[self.flow_control_charge_id, "controlled_mdot_kg_per_s"] = 0
                self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "mdot_flow_kg_per_s"] = abs(self.mass_flow)
                self.net.circ_pump_mass.at[self.circ_pump_decharge_id, "t_flow_k"] = self.T_hot
                self.net.flow_control.at[self.flow_control_decharge_id, "controlled_mdot_kg_per_s"] = abs(self.mass_flow)

    def Estimate_loss_ambient(self, T_amb):
        self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
        self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
        Q_loss_hot = self.UA_hot_layer * (self.T_hot - T_amb)
        Q_loss_cold = self.UA_cold_layer * (self.T_cold - T_amb)
        return Q_loss_hot, Q_loss_cold

    def V_dis(self, dt_s, T_amb, mass_flow):
        self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
        self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
        
        if mass_flow >= 0:
            T_net = self.net.res_circ_pump_mass.at[self.circ_pump_charge_id, "t_from_k"]
            Dv = mass_flow * dt_s / self.fluid.get_density(self.T_hot)
            v_in = max(0.0, min(Dv, self.V_cold - self.V_MIN))
            
            # Balance térmico simple
            cp_hot = self.fluid.get_heat_capacity(self.T_hot)
            rho_hot = self.fluid.get_density(self.T_hot)
            cp_cold = self.fluid.get_heat_capacity(self.T_cold)
            rho_cold = self.fluid.get_density(self.T_cold)
            if v_in < Dv - 1e-6:
                            print(f"  ⚠ [{self.name}] Tanque saturado de calor: "
                                  f"{Dv - v_in:.2f} m3 no se pudieron cargar") 
            if self.direction=='to_Tcold':
                self.T_hot += dt_s * (- ((self.UA_hot_layer * (self.T_hot - T_amb)) / (rho_hot * self.V_hot * cp_hot)))
                self.T_cold += dt_s * (((self.mdot_bypass * (T_net - self.T_cold)) / (rho_hot * self.V_cold)) - ((self.UA_cold_layer * (self.T_cold - T_amb)) / (rho_cold * self.V_cold * cp_cold)))
            else:
                self.T_hot += dt_s * (((mass_flow * (T_net - self.T_hot)) / (rho_hot * self.V_hot)) - ((self.UA_hot_layer * (self.T_hot - T_amb)) / (rho_hot * self.V_hot * cp_hot)))
                self.T_cold += dt_s * (-self.UA_cold_layer * (self.T_cold - T_amb) / (rho_cold * self.V_cold * cp_cold))
                self.V_hot += v_in
                self.V_cold -= v_in
        else:
            T_net = self.net.res_circ_pump_mass.at[self.circ_pump_decharge_id, "t_from_k"]
            Dv = abs(mass_flow) * dt_s / self.fluid.get_density(self.T_cold)
            v_in = max(0.0, min(Dv, self.V_hot - self.V_MIN))
            if v_in < Dv - 1e-6:
                            print(f"  ⚠ [{self.name}] Tanque saturado de frio: "
                                  f"{Dv - v_in:.2f} m3 no se pudieron descargar")
            cp_hot = self.fluid.get_heat_capacity(self.T_hot)
            rho_hot = self.fluid.get_density(self.T_hot)
            cp_cold = self.fluid.get_heat_capacity(self.T_cold)
            rho_cold = self.fluid.get_density(self.T_cold)

            self.T_cold += dt_s * ((abs(mass_flow) * (T_net - self.T_cold) / (rho_cold * self.V_cold)) - (self.UA_cold_layer * (self.T_cold - T_amb) / (rho_cold * self.V_cold * cp_cold)))
            self.T_hot += dt_s * (-(self.UA_hot_layer * (self.T_hot - T_amb) / (rho_hot * self.V_hot * cp_hot)))
            self.V_cold += v_in
            self.V_hot -= v_in

        self.v_hot_fraction = self.V_hot / self.V_tot

In [152]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_cons_ida_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_Ida")
j_cons_ida_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_Ida")
j_cons_ida_3=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_Ida")
j_storage=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Storage")
j_by_pass_supply=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="By_pass_supply")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_cons_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_ret")
j_cons_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_ret")
j_cons_ret_3=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_ret")
j_by_pass_return=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="By_pass_supply")

In [153]:
#Plant-storage
#Planta Nodo
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15), text_k=273.2, name="Tubo_Ida_Plant_Storage_ida",k_mm=0.1*1000
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_RetornoPlant_Storage_ret",k_mm=0.1*1000)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_Storage_ida_nodo_1",k_mm=0.1*1000
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_Storage_ret_nodo_1",k_mm=0.1*1000
)

#Node Consumers 
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_cons_ida_1,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_1_ida",k_mm=0.1*1000
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_1_ret",k_mm=0.1*1000
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_cons_ida_2,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_2_ida",k_mm=0.1*1000
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_2_ret",k_mm=0.1*1000
)
#Node_Cons3
pipe_ida_5= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_cons_ida_3,
    length_km=0.0441, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_3_ida",k_mm=0.1*1000
)
pipe_retorno_5= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_3, to_junction=j_nodo_2_ret,
    length_km=0.0441, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_3_ret",k_mm=0.1*1000
)

In [154]:
#heat consumer 
HC_1=pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_1 ,
    to_junction=j_cons_ret_1,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_1"
)
#Consm2
HC_2=pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_2,
    to_junction=j_cons_ret_2,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_2"
)
#Consm3
HC_3=pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_3,
    to_junction=j_cons_ret_3,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_3")

In [155]:
#Plant: 
Plant_1=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=360 ,name='Grid'
)

In [156]:
#decharging
Circ_pump_decharge=pp.create_circ_pump_const_mass_flow(
    net, return_junction=j_nodo_1_ret, flow_junction=j_storage,
    mdot_flow_kg_per_s=0.1, t_flow_k=300, p_flow_bar=5 , name="Circ_mass_decharge")

Flow_control_decharge=pp.create_flow_control(
    net, from_junction=j_storage, to_junction=j_nodo_1,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,name="Flow_control_decharge"
)
Flow_control_bypass_decharge_go=pp.create_flow_control(
    net, from_junction=j_nodo_1_ret, to_junction=j_by_pass_return,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,
)
Flow_control_bypass_decharge_return=pp.create_flow_control(
    net, from_junction=j_by_pass_return, to_junction=j_nodo_1_ret,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,
)

#Charging 
Circ_pump_charge=pp.create_circ_pump_const_mass_flow(
    net, return_junction=j_nodo_1, flow_junction=j_storage,
    mdot_flow_kg_per_s=0.1, t_flow_k=300, p_flow_bar=5  # valor arbitrario, ver nota abajo
)

Flow_control_charge=pp.create_flow_control(
    net, from_junction=j_storage, to_junction=j_nodo_1_ret,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,
)
Flow_control_bypass_charge_go=pp.create_flow_control(
    net, from_junction=j_nodo_1, to_junction=j_by_pass_supply,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,
)
Flow_control_bypass_charge_return=pp.create_flow_control(
    net, from_junction=j_by_pass_supply, to_junction=j_nodo_1,
    controlled_mdot_kg_per_s=0.1, loss_coefficient=0,
)


In [157]:
Storage=ThermoclineTwoLayer(net, "Storage_1", Circ_pump_charge,Circ_pump_decharge, Flow_control_charge, Flow_control_decharge,Flow_control_bypass_charge_go,Flow_control_bypass_charge_return,Flow_control_bypass_decharge_go,Flow_control_bypass_decharge_return,volume_m3=10, t_hot_init=354, t_cold_init=313, v_hot_fraction_init=2/10,UA_loss=15, UA_interface=0.0)

In [158]:
Data=pd.read_excel("C:\\Users\\sserranose\\OneDrive - INSA Lyon\\Bureau\\Code\\Panda_pipes\\Profiles_For_Energy_Storage.xlsx")

In [159]:
Q_consumer_1=Data.iloc[:,2].to_numpy()
Q_consumer_2=Data.iloc[:,3].to_numpy()
Q_consumer_3=Data.iloc[:,4].to_numpy()
Débit_Storage=Data.iloc[:,5].to_numpy()*-1

In [ ]:

keys=["Q_plant_PD","T_hot_storage_PD", "Q_loss_storage_PD", "Q_Storage_PD", "T_hot_storage_PD","T_cold_storage_PD",'V_hot_PD','V_cold_PD','bypass','direction','mdot_entering','mdot_bypass']
results=my_dict = {key: [] for key in keys}
# Bucle de simulación temporal
dt_seconds = 3600
T_ambient = 273.2

for i in range(len(Débit_Storage)):
    # 1. Configurar los consumidores de calor
    net.heat_consumer.at[HC_1, "in_service"] = Q_consumer_1[i] > 0
    if Q_consumer_1[i] > 0: net.heat_consumer.at[HC_1, "qext_w"] = Q_consumer_1[i]
    
    net.heat_consumer.at[HC_2, "in_service"] = Q_consumer_2[i] > 0
    if Q_consumer_2[i] > 0: net.heat_consumer.at[HC_2, "qext_w"] = Q_consumer_2[i]
    
    net.heat_consumer.at[HC_3, "in_service"] = Q_consumer_3[i] > 0
    if Q_consumer_3[i] > 0: net.heat_consumer.at[HC_3, "qext_w"] = Q_consumer_3[i]

    mdot_target = Débit_Storage[i]

    if mdot_target == 0:
        # Si no hay flujo en el tanque, no requiere evaluación de bypass
        Storage.bypass = False
        Storage.mass_flow = 0
        Storage.apply_control_settings()
        pp.pipeflow(net, mode="bidirectional")
    else:
        # PASO 1: Simulación preliminar sin bypass para leer T_network real
        Storage.bypass = False
        Storage.mass_flow = mdot_target
        Storage.apply_control_settings()
        pp.pipeflow(net, mode="bidirectional")

        # Extraer T_network actual dependiendo si es Carga o Descarga
        if mdot_target >= 0:
            T_network = net.res_circ_pump_mass.at[Storage.circ_pump_charge_id, "t_from_k"]
        else:
            T_network = net.res_circ_pump_mass.at[Storage.circ_pump_decharge_id, "t_from_k"]

        # PASO 2: Evaluar la necesidad de Bypass y Volúmenes
        Storage.evaluate_bypass(T_network=T_network, mass_flow=mdot_target, dt_s=dt_seconds)

        # PASO 3: Simulación definitiva con los controles ajustados
        Storage.apply_control_settings()
        pp.pipeflow(net, mode="bidirectional")

        # PASO 4: Actualizar volúmenes y temperaturas internas del acumulador
        Storage.V_dis(dt_s=dt_seconds, mass_flow=mdot_target, T_amb=T_ambient)

    # Guardar resultados
    Q_loss_hot, Q_loss_cold = Storage.Estimate_loss_ambient(T_amb=T_ambient)
    results["Q_plant_PD"].append(float(net.res_circ_pump_pressure.at[Plant_1, "qext_w"]))
    results["Q_loss_storage_PD"].append(float(Q_loss_hot + Q_loss_cold))
    results["T_hot_storage_PD"].append(float(Storage.T_hot))
    results["T_cold_storage_PD"].append(float(Storage.T_cold))
    
    if mdot_target >= 0:
        results["Q_Storage_PD"].append(float(net.res_circ_pump_mass.at[Circ_pump_charge, "qext_w"]))
    else:
        results["Q_Storage_PD"].append(float(net.res_circ_pump_mass.at[Circ_pump_decharge, "qext_w"]))
        
    results['V_hot_PD'].append(Storage.V_hot)
    results['V_cold_PD'].append(Storage.V_cold)
    results['bypass'].append(Storage.bypass)
    results['direction'].append(Storage.direction)
    results['mdot_entering'].append(Storage.mdot_entering)
    results['mdot_bypass'].append(Storage.mdot_bypass)


0
1
2
3
4
5
6
7
8
9
10


In [161]:
df=pd.DataFrame(results)
df.to_excel("output_real_thermocline.xlsx", index=False, sheet_name="Sheet1")